# example how to generate a feature cube based on EO data (Sentinel-1 and Sentinel-2) for a 20x20km tile
this tests include the generation of the feature cube with and without NVBT band. The NVBT band is a measure of the number of valid input data timesteps after cloud masking and the internal temporal binning.
The NVBT band can be useful for identifying areas with high or low levels of observation, which can be important for a variety of applications, such as monitoring changes in land use or assessing the quality of data in a given area.

In [1]:
from eo_processing.utils.helper import init_connection
from eo_processing.openeo.processing import generate_master_feature_cube
from eo_processing.config.settings import get_advanced_options, get_job_options, get_collection_options

#### declare space and time

In [2]:
# the time context is given by start and end date
year = 2024
start = f'{year}-01-01'
end = f'{year+1}-01-01'   # the end is always exclusive

# the space context is defined as a bounding box dictionary with south,west,north,east and crs
# we take as example an 20x20km tile in EU LAEA grid around Vienna
AOI = {'east': 4800000, 'south': 2820000, 'west': 4780000, 'north': 2840000, 'crs': 3035}

### get processing_options for the eo_processing functions, collection_options and job_options

In [3]:
processing_options = get_advanced_options(provider='cdse', skip_check_S1=True, skip_check_S2=True)
job_options = get_job_options(provider='cdse', task='feature_generation')
collection_options = get_collection_options(provider='cdse')
processing_options.update({'openeo_chunk_size': 64})

In [4]:
processing_options

{'provider': 'cdse',
 's1_orbitdirection': 'DESCENDING',
 'target_crs': 3035,
 'resolution': 10.0,
 'time_interpolation': False,
 'ts_interval': 'dekad',
 'S2_temporal_reducer': 'median',
 'S1_temporal_reducer': 'mean',
 'SLC_masking_algo': 'mask_scl_dilation',
 'S2_max_cloud_cover': 95,
 'S2_bands': ['B02',
  'B03',
  'B04',
  'B05',
  'B06',
  'B07',
  'B08',
  'B8A',
  'B11',
  'B12'],
 's2_tileid_list': None,
 'skip_check_S1': True,
 'skip_check_S2': True,
 'apply_cloud_mask': True,
 'get_NVBT': False,
 'optical_vi_list': ['ABDI1',
  'ABDI2',
  'AWEInsh',
  'AVI',
  'BLFEI',
  'CIRE',
  'EVI',
  'IRECI',
  'MBWI',
  'MNDWI',
  'MNDVI',
  'NDMI',
  'NDVI',
  'NDVIMNDWI',
  'NDWI',
  'NMDI',
  'NIRv',
  'S2WI',
  'S2REP',
  'WRI'],
 'radar_vi_list': ['VHVVD', 'VHVVR', 'DpRVIVV'],
 'S2_scaling': [0, 10000, 0, 1.0],
 'S1_db_rescale': True,
 'append': True,
 'openeo_chunk_size': 64}

### establish connection to openEO

In [5]:
con = init_connection(provider='cdse')

Authenticated using refresh token.


### run the feature cube generation WITHOUT NVBT band

In [6]:
# update job_options due toBerts setting from last inference runs
# ToDO: optimize the job settings for feature_cube_generation_with_nobs & feature_cube_generation and put in settings + add correct task to 'get_job_options'
job_options.update({
    "driver-memory": "8G",
    "driver-memoryOverhead": "4G",
    "executor-memory": "8G",
    "executor-memoryOverhead": "3g",
    "max-executors": 10,
    "python-memory": "disable",
    "allow_empty_cubes": True,
    "soft-errors": True})
   # "soft-errors": 0.05})

In [7]:
# get master cube without nobs
data = generate_master_feature_cube(con, AOI, start, end, **collection_options, **processing_options)

In [ ]:
data.execute_batch(r'C:\Users\buchhorm\Downloads\test_cube\features_cube_v2.tif', title='feature without nobs (20x20km)', job_options=job_options)

0:00:00 Job 'j-260625132115419ba785da40021c22e7': send 'start'
0:00:05 Job 'j-260625132115419ba785da40021c22e7': created (progress 0%)
0:00:10 Job 'j-260625132115419ba785da40021c22e7': queued (progress 0%)
0:00:17 Job 'j-260625132115419ba785da40021c22e7': queued (progress 0%)
0:00:25 Job 'j-260625132115419ba785da40021c22e7': queued (progress 0%)
0:00:35 Job 'j-260625132115419ba785da40021c22e7': queued (progress 0%)
0:00:47 Job 'j-260625132115419ba785da40021c22e7': queued (progress 0%)
0:01:02 Job 'j-260625132115419ba785da40021c22e7': queued (progress 0%)


## run with activated NVBT generation

In [6]:
# now we run same with nobs_perc band
processing_options.update({'get_NVBT': True})
data2 = generate_master_feature_cube(con, AOI, start, end, **collection_options, **processing_options)

In [6]:
# update job_options due to OOM when creating nobs_perc band
# ToDO: optimize the job settings for feature_cube_generation_with_nobs & feature_cube_generation and put in settings + add correct task to 'get_job_options'
job_options.update({
    "driver-memory": "4g",
    "driver-memoryOverhead": "4g",
    "executor-memory": "14g",
    "executor-memoryOverhead": "10g",
    "max-executors": 10,
    "python-memory": "disable",})


In [ ]:
data2.execute_batch(r'C:\Users\buchhorm\Downloads\test_cube\features_cube_with_nobs_v2.tif', title='feature with nobs (20x20km)', job_options=job_options)

0:00:00 Job 'j-2606251209214537a5b84140ae0d5972': send 'start'
0:00:03 Job 'j-2606251209214537a5b84140ae0d5972': queued (progress 0%)
0:00:09 Job 'j-2606251209214537a5b84140ae0d5972': queued (progress 0%)
0:00:15 Job 'j-2606251209214537a5b84140ae0d5972': queued (progress 0%)
0:00:23 Job 'j-2606251209214537a5b84140ae0d5972': queued (progress 0%)
0:00:33 Job 'j-2606251209214537a5b84140ae0d5972': queued (progress 0%)
0:00:46 Job 'j-2606251209214537a5b84140ae0d5972': queued (progress 0%)
0:01:01 Job 'j-2606251209214537a5b84140ae0d5972': running (progress N/A)
0:01:20 Job 'j-2606251209214537a5b84140ae0d5972': running (progress N/A)
0:01:44 Job 'j-2606251209214537a5b84140ae0d5972': running (progress N/A)
0:02:14 Job 'j-2606251209214537a5b84140ae0d5972': running (progress N/A)
0:02:51 Job 'j-2606251209214537a5b84140ae0d5972': running (progress N/A)
0:03:38 Job 'j-2606251209214537a5b84140ae0d5972': running (progress N/A)
0:04:37 Job 'j-2606251209214537a5b84140ae0d5972': running (progress N/A)
